# `langchain` smoke test (langchain 1.x)

langchain 1.x removed the legacy `langchain.chains` / `langchain.memory` /
`langchain.agents.initialize_agent` / `langchain.callbacks` / `langchain.cache`
surfaces. The current package exposes `agents` (create_agent), `chat_models`
(init_chat_model), `embeddings` (init_embeddings), `messages`, `tools`, and
`rate_limiters`. All cells run offline (dummy keys / fake models only).

In [ ]:
# Example 1: message types + trim_messages (the 1.x conversation-history primitive).
from langchain.messages import AIMessage, HumanMessage, SystemMessage, trim_messages

history = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="Hello, what can you help me with?"),
    AIMessage(content="I can help with analysis, writing, and problem-solving."),
    HumanMessage(content="Can you help me write a Python function?"),
    AIMessage(content="Of course! What kind of function?"),
]

# token_counter=len counts each message as 1 -> keep the last 3 messages.
trimmed = trim_messages(history, max_tokens=3, token_counter=len, strategy="last")
print(f"history: {len(history)} messages -> trimmed: {len(trimmed)} messages")
for msg in trimmed:
    print(f"  {type(msg).__name__}: {msg.content[:45]}")

assert len(trimmed) == 3
assert isinstance(trimmed[-1], AIMessage)
print("Example 1 completed successfully")

In [ ]:
# Example 2: the @tool decorator — schema derivation + direct invocation.
from langchain.tools import BaseTool, tool


@tool
def rectangle_area(length: float, width: float) -> float:
    """Calculate the area of a rectangle."""
    return length * width


print(f"tool name: {rectangle_area.name}")
print(f"tool description: {rectangle_area.description}")
print(f"args schema: {rectangle_area.args}")

result = rectangle_area.invoke({"length": 3.0, "width": 4.0})
print(f"rectangle_area(3, 4) = {result}")

assert isinstance(rectangle_area, BaseTool)
assert result == 12.0
assert set(rectangle_area.args) == {"length", "width"}
print("Example 2 completed successfully")

In [ ]:
# Example 3: chat models — init_chat_model provider inference (offline, dummy key)
# + a real invoke round-trip through a fake chat model.
from langchain.chat_models import BaseChatModel, init_chat_model
from langchain.messages import AIMessage, HumanMessage
from langchain_core.language_models import GenericFakeChatModel

model = init_chat_model("openai:gpt-4o-mini", api_key="sk-dummy-key-offline")
print(f"init_chat_model -> {type(model).__name__}")
assert isinstance(model, BaseChatModel)

fake = GenericFakeChatModel(messages=iter([AIMessage(content="fake reply")]))
reply = fake.invoke([HumanMessage(content="hello")])
print(f"fake invoke -> {type(reply).__name__}: {reply.content!r}")

assert isinstance(reply, AIMessage)
assert reply.content == "fake reply"
print("Example 3 completed successfully")

In [ ]:
# Example 4: create_agent — build a LangGraph agent and run it end-to-end offline
# with a scripted fake model (no tools -> the model node answers directly).
from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage
from langchain_core.language_models import GenericFakeChatModel

fake_model = GenericFakeChatModel(messages=iter([AIMessage(content="agent says done")]))
agent = create_agent(fake_model, tools=[])
print(f"agent type: {type(agent).__name__}")

nodes = list(agent.get_graph().nodes)
print(f"graph nodes: {nodes}")
assert "model" in nodes

result = agent.invoke({"messages": [HumanMessage(content="run the task")]})
final = result["messages"][-1]
print(f"final message: {type(final).__name__}: {final.content!r}")

assert isinstance(final, AIMessage)
assert final.content == "agent says done"
print("Example 4 completed successfully")

In [ ]:
# Example 5: embeddings — init_embeddings factory (offline, dummy key) + a
# deterministic embedding round-trip invariant.
from langchain.embeddings import Embeddings, init_embeddings
from langchain_core.embeddings import DeterministicFakeEmbedding

emb = init_embeddings("openai:text-embedding-3-small", api_key="sk-dummy-key-offline")
print(f"init_embeddings -> {type(emb).__name__}")
assert isinstance(emb, Embeddings)

fake = DeterministicFakeEmbedding(size=16)
v1 = fake.embed_query("hello world")
v2 = fake.embed_query("hello world")
docs = fake.embed_documents(["alpha", "beta"])

print(f"embedding dim: {len(v1)}; documents embedded: {len(docs)}")
assert len(v1) == 16 and v1 == v2  # deterministic: same text -> same vector
assert len(docs) == 2 and docs[0] != docs[1]
print("Example 5 completed successfully")